In [1]:
import pandas as pd
import regex as re
from sklearn.preprocessing import MinMaxScaler

# Data Cleaning

### Cleaning Function

In [5]:
scaler = MinMaxScaler()
def clean_rating(ori_df):
    df = ori_df.copy()
    extracted_data = df['rating'].str.extract(
        r'Score: (\d+\.?\d*) \(scored by ([\d,]+) users\)')
    df['score'] = pd.to_numeric(extracted_data[0], errors='coerce')
    df['user_count'] = pd.to_numeric(extracted_data[1].str.replace(
        ',', ''), errors='coerce', downcast='integer')

    df['score'] = df['score'].fillna(0)
    df['user_count'] = df['user_count'].fillna(0)

    
    df['rating_norm'] = scaler.fit_transform(df[['score']])

    return df

In [ ]:
def clean_text(ori_df):
    df = ori_df.copy()

    # Clean missing values
    df['genre'] = df['genre'].fillna("")
    # Clean synopsis
    pattern = r'\s*\(?Edit Translation\)?\s*$'
    df['synopsis'] = df['synopsis'].apply(
        lambda x: x.split("(Source", 1)[0].strip())
    df['synopsis'] = df['synopsis'].str.replace(
        pattern, '', regex=True, case=False)

    # Clean durations
    df['hour'] = df['duration'].str.extract(
        r'(\d+)\s*hr', expand=False).fillna(0).astype(int)
    df['minutes'] = df['duration'].str.extract(
        r'(\d+)\s*min', expand=False).fillna(0).astype(int)
    df['duration (min)'] = df['hour'] * 60 + df['minutes']
    df.drop(['hour', 'minutes'], axis=1, inplace=True)
    df[['duration', 'duration (min)']].head()

    # Clean release date
    df['release_date'] = df['release_date'].apply(
        lambda x: x.split("-", 1)[0].strip())
    df['release_year'] = df['release_date'].str.extract(r'(\d{4})')

    # Maping rating to numeric
    rating_map = {
        "G - All Ages": 0,
        "13+ - Teens 13 or older": 13,
        "15+ - Teens 15 or older": 15,
        "18+ Restricted (violence & profanity)": 18,
        "R - Restricted Screening (nudity & violence)": 21
    }
    df['rate_num'] = df['ct_rate'].map(rating_map).fillna(0).astype(int)

    # Optional
    df = df[~df['tag'].str.contains("lesbian", case=False, na=False)]

    return df

### Rating

In [7]:
drama = pd.read_csv('data/raw/rating_drama.csv')
movies = pd.read_csv('data/raw/rating_movies.csv')
shows = pd.read_csv('data/raw/rating_kshows.csv')

print(f"Drama: {drama.shape}")
print(f"Drama: {movies.shape}")
print(f"Drama: {shows.shape}")

Drama: (3628, 2)
Drama: (1613, 2)
Drama: (2295, 2)


In [8]:
r_drama = clean_rating(drama)
r_movies = clean_rating(movies)
r_shows = clean_rating(shows)

print(f"Drama: {r_drama.shape}")
print(f"Drama: {r_movies.shape}")
print(f"Drama: {r_shows.shape}")

Drama: (3628, 5)
Drama: (1613, 5)
Drama: (2295, 5)


In [9]:
r_drama.head()

,drama_id,rating,score,user_count,rating_norm
0,735043,"Score: 9.3 (scored by 68,706 users)",9.3,68706.0,0.93
1,739603,"Score: 9.2 (scored by 103,873 users)",9.2,103873.0,0.92
2,49231,"Score: 9.1 (scored by 75,160 users)",9.1,75160.0,0.91
3,702267,"Score: 9.1 (scored by 114,265 users)",9.1,114265.0,0.91
4,52939,"Score: 9.1 (scored by 108,485 users)",9.1,108485.0,0.91


### Metadata Cleaning

In [10]:
m_drama = pd.read_csv('data/raw/metadata_drama.csv')
m_movies = pd.read_csv('data/raw/metadata_movie.csv')
m_shows = pd.read_csv('data/raw/metadata_variety.csv')

print(f"Drama: {m_drama.shape}")
print(f"Movies: {m_movies.shape}")
print(f"Shows: {m_shows.shape}")
m_shows.head(2)

Drama: (3624, 14)
Movies: (1609, 14)
Shows: (2295, 14)


,id,title,native-title,genre,director,format,type,country,release_date,duration,episodes,ct_rate,tag,synopsis
0,808242,EXO's Travel the World on a Ladder Season 5,EXO의 사다리타고 세계여행 시즌5 : 제주편,"Adventure, Comedy",NaN,Variety Show,TV Program,South Korea,"Mar 4, 2026 - Apr 8, 2026",1 hr. 15 min.,6,13+ - Teens 13 or older,"EXO-CBX, EXO, Friendship (Vote tags)","Set on the beautiful island of Jeju, “EXO’s La..."
1,805776,When Our Kids Fall in Love Season 2,내 새끼의 연애 시즌2,Romance,NaN,Reality Program,TV Program,South Korea,"Feb 25, 2026 - Apr 29, 2026",1 hr. 30 min.,10,15+ - Teens 15 or older,"Cohabitation, Dating Show, Reality Show, Frien...",Love creates faces even parents see for the fi...


In [11]:
cl_drama = clean_text(m_drama)
cl_drama = cl_drama[~cl_drama['title'].str.contains('Drama Special')]
cl_movies = clean_text(m_movies)
cl_shows = clean_text(m_shows)

print(f"Drama: {cl_drama.shape}")
print(f"Movies: {cl_movies.shape}")
print(f"Shows: {cl_shows.shape}")
cl_movies.head()

Drama: (3349, 17)
Movies: (1609, 17)
Shows: (2295, 17)


,id,title,native-title,genre,director,format,type,country,release_date,duration,episodes,ct_rate,tag,synopsis,duration (min),release_year,rate_num
0,785650,Salmokji: Whispering Water,살목지,"Thriller, Mystery, Horror",NaN,Feature Film,Movie,South Korea,"Apr 8, 2026",1 hr. 35 min.,NaN,15+ - Teens 15 or older,"Producer Supporting Character, Brothers' Relat...",What lurks beneath the reservoir awaits those ...,95,2026,15
1,767215,Pavane,파반느,"Romance, Drama, Melodrama",NaN,Feature Film,Movie,South Korea,"Feb 20, 2026",1 hr. 53 min.,NaN,15+ - Teens 15 or older,"Ostracized Female Lead, Misfit Female Lead, De...",The moving love story of a man loved by everyo...,113,2026,15
2,771865,Humint,휴민트,"Action, Thriller, Romance, Crime",NaN,Feature Film,Movie,South Korea,"Feb 11, 2026",1 hr. 59 min.,NaN,15+ - Teens 15 or older,"Agent Male Lead, Spy Male Lead, National Intel...",The story depicts secret agents from North and...,119,2026,15
3,783666,Number One,넘버원,"Drama, Fantasy",Kim Tae Yong,Feature Film,Movie,South Korea,"Feb 11, 2026",1 hr. 45 min.,NaN,13+ - Teens 13 or older,"Healing, Mother-Son Relationship (Vote tags)",Eighteen-year-old Ha Min suddenly begins to se...,105,2026,13
4,774369,The King’s Warden,왕과 사는 남자,"Historical, Drama",Jang Hang Joon,Feature Film,Movie,South Korea,"Feb 4, 2026",1 hr. 57 min.,NaN,13+ - Teens 13 or older,"1400s, Exiled Male Lead, Village Chief Male Le...",Fate entwines a fallen king with the man who b...,117,2026,13


## Data Merge

In [12]:
mg_drama = pd.merge(cl_drama,r_drama,"left",left_on='id', right_on='drama_id')
mg_movies = pd.merge(cl_movies, r_movies, "left", left_on='id', right_on='drama_id')
mg_shows = pd.merge(cl_shows, r_shows, "left", left_on='id', right_on='drama_id')

print(mg_drama.shape)
print(mg_movies.shape)
print(mg_shows.shape)
mg_drama.head(3)

(3349, 22)
(1609, 22)
(2295, 22)


,id,title,native-title,genre,director,format,type,country,release_date,duration,...,tag,synopsis,duration (min),release_year,rate_num,drama_id,rating,score,user_count,rating_norm
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,"Family Relationship, Nice Male Lead, Healthy M...",It is a story that resembles a tribute to our ...,62,2025,13,735043,"Score: 9.3 (scored by 68,706 users)",9.3,68706.0,0.93
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,"Time Travel, Child From The Future, Deafness, ...","In 2023, high school student Eun Gyeol, a CODA...",70,2023,15,739603,"Score: 9.2 (scored by 103,873 users)",9.2,103873.0,0.92
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,"Uncle-Nephew Relationship, Autism, Death, Tear...",Han Geu Ru is an autistic 20-year-old guy. He ...,52,2021,18,49231,"Score: 9.1 (scored by 75,160 users)",9.1,75160.0,0.91


In [13]:
mg_drama.drop(['drama_id', 'rating'], axis=1, inplace=True)
mg_movies.drop(['drama_id', 'rating'], axis=1, inplace= True)
mg_shows.drop(['drama_id', 'rating'], axis=1, inplace= True)

print(mg_drama.shape)
print(mg_movies.shape)
print(mg_shows.shape)
mg_movies.head(3)

(3349, 20)
(1609, 20)
(2295, 20)


,id,title,native-title,genre,director,format,type,country,release_date,duration,episodes,ct_rate,tag,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm
0,785650,Salmokji: Whispering Water,살목지,"Thriller, Mystery, Horror",NaN,Feature Film,Movie,South Korea,"Apr 8, 2026",1 hr. 35 min.,NaN,15+ - Teens 15 or older,"Producer Supporting Character, Brothers' Relat...",What lurks beneath the reservoir awaits those ...,95,2026,15,7.0,244.0,0.70
1,767215,Pavane,파반느,"Romance, Drama, Melodrama",NaN,Feature Film,Movie,South Korea,"Feb 20, 2026",1 hr. 53 min.,NaN,15+ - Teens 15 or older,"Ostracized Female Lead, Misfit Female Lead, De...",The moving love story of a man loved by everyo...,113,2026,15,8.0,4738.0,0.80
2,771865,Humint,휴민트,"Action, Thriller, Romance, Crime",NaN,Feature Film,Movie,South Korea,"Feb 11, 2026",1 hr. 59 min.,NaN,15+ - Teens 15 or older,"Agent Male Lead, Spy Male Lead, National Intel...",The story depicts secret agents from North and...,119,2026,15,7.7,1949.0,0.77


## Saving

In [14]:
mg_drama.to_csv('data/clean/c_drama.csv',index=False)
mg_movies.to_csv('data/clean/c_movies.csv', index=False)
mg_shows.to_csv('data/clean/c_shows.csv', index=False)

# Data Preparation

## Cast Data merge

In [15]:
cast_drama = pd.read_csv('data/raw/cast_drama.csv')
cast_movies = pd.read_csv('data/raw/cast_movie.csv')
cast_shows = pd.read_csv('data/raw/cast_variety.csv')

print(cast_drama.shape)
print(cast_movies.shape)
print(cast_shows.shape)
cast_drama.head()

(18135, 3)
(7658, 3)
(12005, 3)


,drama_id,actor,role
0,735043,IU,Main Role
1,735043,Park Bo Gum,Main Role
2,735043,Moon So Ri,Main Role
3,735043,Park Hae Joon,Main Role
4,735043,Kim Yong Rim,Support Role


In [16]:
cg_drama =  (cast_drama.groupby('drama_id')['actor'].agg(", ".join).reset_index())
cg_movies = (cast_movies.groupby('drama_id')['actor'].agg(", ".join).reset_index())
cg_shows = (cast_shows.groupby('drama_id')['actor'].agg(", ".join).reset_index())

print(cg_drama.shape)
print(cg_movies.shape)
print(cg_shows.shape)

cg_drama.head()

(3375, 2)
(1532, 2)
(2273, 2)


,drama_id,actor
0,8,"Hyun Bin, Ha Ji Won, Yoon Sang Hyun, Kim Sa Ra..."
1,9,"Hyun Bin, Kim Min Joon, Seo Do Young, Wang Ji ..."
2,12,"Jang Hyuk, Lee Min Jung, Noh Min Woo, Kim Hee ..."
3,19,"Lee Min Ho, Son Ye Jin, Kim Ji Suk, Wang Ji Hy..."
4,20,"Kim Hyun Joong, Jung So Min, Lee Tae Sung, Hon..."


In [17]:
## Merge the metadata and cast
f_drama = mg_drama.merge(cg_drama, 'left', left_on='id', right_on='drama_id')
f_movies = mg_movies.merge(cg_movies, 'left', left_on='id', right_on='drama_id')
f_shows = mg_shows.merge(cg_shows, 'left', left_on='id', right_on='drama_id')

print(f_drama.shape)
print(f_movies.shape)
print(f_shows.shape)
f_drama.head()

(3349, 22)
(1609, 22)
(2295, 22)


,id,title,native-title,genre,director,format,type,country,release_date,duration,...,tag,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm,drama_id,actor
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,"Family Relationship, Nice Male Lead, Healthy M...",It is a story that resembles a tribute to our ...,62,2025,13,9.3,68706.0,0.93,735043.0,"IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Ki..."
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,"Time Travel, Child From The Future, Deafness, ...","In 2023, high school student Eun Gyeol, a CODA...",70,2023,15,9.2,103873.0,0.92,739603.0,"Ryeo Un, Choi Hyun Wook, Seol In Ah, Shin Eun ..."
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,"Uncle-Nephew Relationship, Autism, Death, Tear...",Han Geu Ru is an autistic 20-year-old guy. He ...,52,2021,18,9.1,75160.0,0.91,49231.0,"Lee Je Hoon, Tang Jun Sang, Hong Seung Hee, Ju..."
3,702267,Weak Hero Class 1,약한영웅 Class 1,"Action, Youth, Drama",Park Dhan Hee,Standard Series,Drama,South Korea,"Nov 18, 2022",40 min.,...,"Student Male Lead, Teenager Male Lead, Violenc...",Yeon Shi Eun is a model student who ranks at t...,40,2022,18,9.1,114265.0,0.91,702267.0,"Park Ji Hoon, Choi Hyun Wook, Hong Kyung, Kim ..."
4,52939,Alchemy of Souls,환혼,"Action, Historical, Romance, Fantasy",Park Joon Hwa,Standard Series,Drama,South Korea,"Jun 18, 2022",1 hr. 20 min.,...,"Transmigration, Master-Disciple Relationship, ...","In the fictional country of Daeho, young mages...",80,2022,15,9.1,108485.0,0.91,52939.0,"Lee Jae Wook, Jung So Min, Hwang Min Hyun, Shi..."


In [18]:
text_columns = [
    "native-title",
    "genre",
    "duration",
    "actor",
    "director",
    "synopsis"
]

## cleaning
f_drama.drop('drama_id',axis=1, inplace=True)
f_movies.drop('drama_id', axis=1, inplace=True)
f_shows.drop('drama_id', axis=1, inplace=True)

## fill missing value
print(f"drama: {f_drama['actor'].isna().sum()}")
print(f"movies: {f_movies['actor'].isna().sum()}")

for column in text_columns:
    f_drama[column] = f_drama[column].fillna("")
    f_movies[column] = f_movies[column].fillna("")
    f_shows[column] = f_shows[column].fillna("")

## fill missing

print(f"drama: {f_drama['actor'].isna().sum()}")
print(f"movies: {f_movies['actor'].isna().sum()}")

drama: 234
movies: 77
drama: 0
movies: 0


In [19]:
concat_df = pd.concat([f_drama, f_movies], ignore_index=True, axis=0)
print(concat_df.shape)
concat_df.head(3)

(4958, 21)


,id,title,native-title,genre,director,format,type,country,release_date,duration,...,ct_rate,tag,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm,actor
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,13+ - Teens 13 or older,"Family Relationship, Nice Male Lead, Healthy M...",It is a story that resembles a tribute to our ...,62,2025,13,9.3,68706.0,0.93,"IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Ki..."
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,15+ - Teens 15 or older,"Time Travel, Child From The Future, Deafness, ...","In 2023, high school student Eun Gyeol, a CODA...",70,2023,15,9.2,103873.0,0.92,"Ryeo Un, Choi Hyun Wook, Seol In Ah, Shin Eun ..."
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,18+ Restricted (violence & profanity),"Uncle-Nephew Relationship, Autism, Death, Tear...",Han Geu Ru is an autistic 20-year-old guy. He ...,52,2021,18,9.1,75160.0,0.91,"Lee Je Hoon, Tang Jun Sang, Hong Seung Hee, Ju..."


## Text Preparation

In [20]:
## Optional function
def clean_tfidf(text):
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r'\(Vote tags\)', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [21]:
def clean_embedding(text):
    if pd.isna(text): return ""
    text = re.sub(r'\(Vote tags\)', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [22]:
def combined_text_emb(row):
    parts = [
        str(row['title']),
        str(row['genre']), str(row['genre']),
        str(row['tag']),
        str(row['director']),
        str(row['actor']),
        str(row['synopsis'])
    ]
    return " ".join([p for p in parts if p])

In [23]:
def combineed_text_tf(row):
    parts = [
        str(row['title']),
        str(row['genre']), str(row['genre']),
        str(row['tag']),
        str(row['director']),
        str(row['actor'])
    ]
    return " ".join([p for p in parts if p])

In [24]:
f_shows['combined_emb'] = f_shows.apply(combined_text_emb, axis=1).apply(clean_embedding)
f_shows['combined_tf'] = f_shows.apply(combineed_text_tf, axis=1).apply(clean_tfidf)
f_shows.head(2)

,id,title,native-title,genre,director,format,type,country,release_date,duration,...,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm,actor,combined_emb,combined_tf
0,808242,EXO's Travel the World on a Ladder Season 5,EXO의 사다리타고 세계여행 시즌5 : 제주편,"Adventure, Comedy",,Variety Show,TV Program,South Korea,"Mar 4, 2026",1 hr. 15 min.,...,"Set on the beautiful island of Jeju, “EXO’s La...",75,2026,13,8.5,44.0,0.85,"Suho, Doh Kyung Soo, Park Chan Yeol, Kai, Oh S...",EXO's Travel the World on a Ladder Season 5 Ad...,exo s travel the world on a ladder season 5 ad...
1,805776,When Our Kids Fall in Love Season 2,내 새끼의 연애 시즌2,Romance,,Reality Program,TV Program,South Korea,"Feb 25, 2026",1 hr. 30 min.,...,Love creates faces even parents see for the fi...,90,2026,15,7.6,213.0,0.76,"Kim Sung Joo, Lee Jong Hyuk, Yoon Min Soo, Par...",When Our Kids Fall in Love Season 2 Romance Ro...,when our kids fall in love season 2 romance ro...


In [25]:
concat_df['combined_emb'] = concat_df.apply(combined_text_emb, axis=1).apply(clean_embedding)
concat_df['combined_tf'] = concat_df.apply(combineed_text_tf, axis=1).apply(clean_tfidf)
concat_df.head(3)

,id,title,native-title,genre,director,format,type,country,release_date,duration,...,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm,actor,combined_emb,combined_tf
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,It is a story that resembles a tribute to our ...,62,2025,13,9.3,68706.0,0.93,"IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Ki...","When Life Gives You Tangerines Romance, Life, ...",when life gives you tangerines romance life dr...
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,"In 2023, high school student Eun Gyeol, a CODA...",70,2023,15,9.2,103873.0,0.92,"Ryeo Un, Choi Hyun Wook, Seol In Ah, Shin Eun ...","Twinkling Watermelon Romance, Youth, Drama, Fa...",twinkling watermelon romance youth drama fanta...
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,Han Geu Ru is an autistic 20-year-old guy. He ...,52,2021,18,9.1,75160.0,0.91,"Lee Je Hoon, Tang Jun Sang, Hong Seung Hee, Ju...","Move to Heaven Life, Drama Life, Drama Uncle-N...",move to heaven life drama life drama uncle nep...


In [26]:
print(concat_df['combined_emb'][0])

When Life Gives You Tangerines Romance, Life, Drama Romance, Life, Drama Family Relationship, Nice Male Lead, Healthy Mains’ Relationship, Break Up, Teenage Pregnancy, Poor Family, Father-Daughter Relationship, Poor Male Lead, Chasing A Dream, Heartwarming Kim Won Suk IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Kim Yong Rim, Na Moon Hee It is a story that resembles a tribute to our parents' tender and still youthful seasons when they were so young, including the story of mother's first love, father's heroic tales, grandma's rebellious youth, and grandpa's romantic era. Ae Sun is rebellious but gets nervous every time she rebels. She is not well off but is always shining and full of positivity. She dreams of becoming a poet, although she cannot attend school, and she is a bold character who doesn’t hide any emotions. Gwan Sik is an extremely diligent and quiet character. Romance is not a strength for him, and he doesn’t know how to act if Ae Sun cries or laughs, but he is a silent warri

In [ ]:
# concat_df.to_csv('data/clean/movies_drama_v2.csv', index=False)
# f_shows.to_csv('data/clean/final_shows.csv')

# Data Final

In [5]:
poster_drama = pd.read_csv('data/raw/poster_drama.csv')
poster_movies = pd.read_csv('data/raw/poster_movies.csv')

print(f"Drama: {poster_drama.shape[0]}")
print(f"Movies: {poster_movies.shape[0]}")

poster_drama.head()

Drama: 3626
Movies: 1611


,drama_id,poster
0,735043,https://i.mydramalist.com/5v8b2y_4c.jpg?v=1
1,739603,https://i.mydramalist.com/2w44jE_4c.jpg?v=1
2,49231,https://i.mydramalist.com/Rle36_4c.jpg?v=1
3,702267,https://i.mydramalist.com/pq2lr_4c.jpg?v=1
4,52939,https://i.mydramalist.com/Beg4z_4c.jpg?v=1


In [7]:
poster_all = pd.concat([poster_drama, poster_movies], axis=0)
print(f"Poster all: {poster_all.shape}")

Poster all: (5237, 2)


In [13]:
data = pd.read_csv('data/clean/movies_drama_v2.csv')
data.head(3)

,id,title,native-title,genre,director,format,type,country,release_date,duration,...,synopsis,duration (min),release_year,rate_num,score,user_count,rating_norm,actor,combined_emb,combined_tf
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,It is a story that resembles a tribute to our ...,62,2025,13,9.3,68706.0,0.93,"IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Ki...","When Life Gives You Tangerines Romance, Life, ...",when life gives you tangerines romance life dr...
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,"In 2023, high school student Eun Gyeol, a CODA...",70,2023,15,9.2,103873.0,0.92,"Ryeo Un, Choi Hyun Wook, Seol In Ah, Shin Eun ...","Twinkling Watermelon Romance, Youth, Drama, Fa...",twinkling watermelon romance youth drama fanta...
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,Han Geu Ru is an autistic 20-year-old guy. He ...,52,2021,18,9.1,75160.0,0.91,"Lee Je Hoon, Tang Jun Sang, Hong Seung Hee, Ju...","Move to Heaven Life, Drama Life, Drama Uncle-N...",move to heaven life drama life drama uncle nep...


In [14]:
processed_moviedrama = pd.merge(data, poster_all, left_on='id', right_on='drama_id', how='left')
processed_moviedrama.drop('drama_id', axis=1, inplace=True)
print(f"Before: {data.shape}")
print(f"After: {processed_moviedrama.shape}")

processed_moviedrama.head(3)

Before: (4958, 23)
After: (4958, 24)


,id,title,native-title,genre,director,format,type,country,release_date,duration,...,duration (min),release_year,rate_num,score,user_count,rating_norm,actor,combined_emb,combined_tf,poster
0,735043,When Life Gives You Tangerines,폭싹 속았수다,"Romance, Life, Drama",Kim Won Suk,Standard Series,Drama,South Korea,"Mar 7, 2025",1 hr. 2 min.,...,62,2025,13,9.3,68706.0,0.93,"IU, Park Bo Gum, Moon So Ri, Park Hae Joon, Ki...","When Life Gives You Tangerines Romance, Life, ...",when life gives you tangerines romance life dr...,https://i.mydramalist.com/5v8b2y_4c.jpg?v=1
1,739603,Twinkling Watermelon,반짝이는 워터멜론,"Romance, Youth, Drama, Fantasy",Son Jung Hyun,Standard Series,Drama,South Korea,"Sep 25, 2023",1 hr. 10 min.,...,70,2023,15,9.2,103873.0,0.92,"Ryeo Un, Choi Hyun Wook, Seol In Ah, Shin Eun ...","Twinkling Watermelon Romance, Youth, Drama, Fa...",twinkling watermelon romance youth drama fanta...,https://i.mydramalist.com/2w44jE_4c.jpg?v=1
2,49231,Move to Heaven,무브 투 헤븐: 나는 유품정리사입니다,"Life, Drama",Kim Sung Ho,Standard Series,Drama,South Korea,"May 14, 2021",52 min.,...,52,2021,18,9.1,75160.0,0.91,"Lee Je Hoon, Tang Jun Sang, Hong Seung Hee, Ju...","Move to Heaven Life, Drama Life, Drama Uncle-N...",move to heaven life drama life drama uncle nep...,https://i.mydramalist.com/Rle36_4c.jpg?v=1


In [15]:
##Saving
processed_moviedrama.to_csv('data/clean/processed_moviesdrama.csv', index=False)